In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

plt.rcParams.update({
    "font.family": "Malgun Gothic", "axes.unicode_minus": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#D8D8DE", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": "#EDEDF2", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "xtick.color": "#5A5A66", "ytick.color": "#5A5A66",
    "axes.labelcolor": "#2A2A33", "text.color": "#2A2A33", "font.size": 11,
})

# 검증 통과 팔레트 (CVD ΔE 12.1 / 대비 3:1+)
C1, C2, C3 = "#5B4FCF", "#B5650A", "#0E8C7A"   # 유형 ① ② ③
INK, MUTED, FAINT = "#2A2A33", "#5A5A66", "#B8B8C4"
유형색 = {1: C1, 2: C2, 3: C3}
유형명 = {1: "① 지역 전체 기회", 2: "② 업종 간 이동", 3: "③ 채널 이동"}

그림 = "data/그림_결과물"; os.makedirs(그림, exist_ok=True)
def 저장(fig, name):
    fig.savefig(f"{그림}/{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
    print("저장:", name)


## 1. 준비 — BC 소비 · 인구 · 점포 수를 한 테이블로

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import KFold

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

plt.rcParams["font.family"] = "Malgun Gothic"      # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

# 색은 역할별로 한 번만 정의한다
파랑, 주황 = "#2a78d6", "#eb6834"
잉크, 보조, 흐림 = "#0b0b0b", "#52514e", "#898781"
격자, 축선, 바탕 = "#e1e0d9", "#c3c2b7", "#fcfcfb"

def 기본축(ax, 격자축="both"):
    # 테두리·눈금을 흐리게 해서 데이터가 주인공이 되게 한다
    ax.set_facecolor(바탕)
    ax.tick_params(which="both", colors=흐림, labelsize=9.5, length=0)
    if 격자축:
        ax.grid(axis=격자축, color=격자, lw=0.8)
    ax.set_axisbelow(True)
    for s_ in ["top", "right"]:
        ax.spines[s_].set_visible(False)
    for s_ in ["left", "bottom"]:
        ax.spines[s_].set_color(축선)

def 약칭(지역):
    # 그림 라벨용 짧은 이름: "전북특별자치도 전주시 완산구" → "전주 완산구"
    t = 지역.split()
    if len(t) == 1:
        return t[0][:2]
    if len(t) == 3:
        return t[1].rstrip("시") + " " + t[2]
    if t[1] in ("중구", "동구", "서구", "남구", "북구"):
        return t[0][:2] + " " + t[1]
    return t[1][:-1] if t[1].endswith("시") else t[1]

def 라벨배치(ax, 좌표들, 글자들, 피할점=(), fontsize=8.5):
    # 점 옆에 라벨을 붙이되, 먼저 붙인 라벨이나 다른 점과 겹치면 위·아래·왼쪽으로 옮겨 본다
    from matplotlib.transforms import Bbox
    fig = ax.figure
    fig.canvas.draw()
    렌더러 = fig.canvas.get_renderer()
    좌표들 = list(좌표들)
    후보위치 = [(9, 0, "left"), (-9, 0, "right"), (9, 11, "left"), (9, -11, "left"),
             (-9, 11, "right"), (-9, -11, "right"), (9, 22, "left"), (9, -22, "left")]
    # 점 자체도 장애물 — 화면 좌표로 바꿔 반지름 8px 상자로 둔다
    놓인것 = []
    for x, y in 좌표들 + list(피할점):
        px, py = ax.transData.transform((x, y))
        놓인것.append(Bbox.from_extents(px - 8, py - 8, px + 8, py + 8))
    for (x, y), 글 in zip(좌표들, 글자들):
        for dx, dy, 정렬 in 후보위치:
            a = ax.annotate(글, (x, y), textcoords="offset points", xytext=(dx, dy), ha=정렬,
                            va="center", fontsize=fontsize, color=잉크, zorder=6)
            상자 = a.get_window_extent(렌더러).expanded(1.0, 1.1)
            if not any(상자.overlaps(b) for b in 놓인것):
                break
            a.remove()
        else:
            a = ax.annotate(글, (x, y), textcoords="offset points", xytext=(7, 0),
                            va="center", fontsize=fontsize, color=잉크, zorder=6)
            상자 = a.get_window_extent(렌더러)
        놓인것.append(상자)

In [ ]:
# BC 소비데이터 — 코드값은 문자열로 고정해야 조인할 때 타입이 맞는다
bc = pd.read_csv("data/ABP_CONTEST_DATA.csv", encoding="utf-8",
                 dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "TP_BUZ_NO": str})

# 인구 — population.ipynb 결과물
pop = pd.read_csv("data/인구_시군구_성별_연령_202601_202606.csv", encoding="utf-8-sig",
                  dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "행정구역코드": str})

# 시군구 × 업종 점포 수 — 상가정보 API 수집 캐시 (광주·전남 제외 227개 시군구, 8개 업종)
업소 = pd.read_csv("data/cache/업소수_전국시군구.csv", encoding="utf-8-sig", dtype={"TP_BUZ_NO": str})

print("BC  :", bc.shape)
print("인구:", pop.shape)
print("점포:", 업소.shape, f"({업소['행정구역명'].nunique()}개 시군구, {업소['TP_BUZ_NO'].nunique()}개 업종)")

In [ ]:
# BC는 시도·시군구가 두 칸이라 인구 데이터처럼 한 칸으로 합친다
bc["행정구역명"] = (bc["SIDO_NM"].str.strip() + " " + bc["CCG_NM"].str.strip()).str.replace(r"\s+", " ", regex=True)
bc["행정구역명"] = bc["행정구역명"].replace("세종특별자치시 세종특별자치시", "세종특별자치시")

# 인천 행정구역 개편 보정 — 점포 캐시는 개편 후 체계라 중구·동구를 합친 단위만 대응된다
인천병합 = {"인천광역시 중구": "인천광역시 중구+동구", "인천광역시 동구": "인천광역시 중구+동구"}
bc["행정구역명"] = bc["행정구역명"].replace(인천병합)
pop["행정구역명"] = pop["행정구역명"].replace(인천병합)

print("점포 캐시에 있는데 BC에 없는 지역:", sorted(set(업소["행정구역명"]) - set(bc["행정구역명"])))
print("점포 캐시에 있는데 인구에 없는 지역:", sorted(set(업소["행정구역명"]) - set(pop["행정구역명"])))

In [ ]:
분석업종 = sorted(업소["TP_BUZ_NO"].unique())      # 8개 — 대형할인점·갈비전문점·한정식은 캐시에 없다
성인 = ["2", "3", "4", "5", "6"]                  # 20대 이상 (0~19세는 카드를 거의 안 쓴다)

# 분자: BC 소비 — 내국인 + 외국인, 법인만 제외
#   점포 수가 내·외국인 구분 없는 전체 점포이므로 분자도 범위를 맞춘다
소비 = (bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & bc["TP_BUZ_NO"].isin(분석업종)]
      .groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)["amt"].sum())

# 인구: 20대 이상, 6개월 평균
인구 = (pop[pop["AGE_CD"].isin(성인)]
      .groupby(["행정구역명", "STRD_YYMM"])["인구"].sum()
      .groupby("행정구역명").mean().rename("성인인구").reset_index())

분석 = (소비.merge(업소, on=["행정구역명", "TP_BUZ_NO"], how="inner")
      .merge(인구, on="행정구역명", how="inner"))
분석 = 분석[(분석["업소수"] > 0) & (분석["amt"] > 0)].copy()     # 로그를 씌울 수 없는 0 제외

print(f"분석 테이블: {len(분석):,}개 조합 / {분석['행정구역명'].nunique()}개 시군구 / {분석['TP_BUZ_NO'].nunique()}개 업종")
display(분석.head())

## 2. 회귀 — 침투지수와 격차금액

업종별로 `log(BC소비) ~ log(점포 수) + log(성인인구)`를 적합해 "이 지역·업종이라면 BC 소비가 얼마 나와야 하는가"를 예측한다.

- **침투지수** = 실제 ÷ 예측 × 100 — 100이면 예측대로, 50이면 절반만 결제
- **격차금액** = 예측 − 실제 — 놓치고 있는 돈
- **시장 백분위** = 예측값의 업종 내 순위 — 업종마다 규모가 10배 이상 달라 업종 안에서 비교한다

In [ ]:
분석["log_소비"] = np.log(분석["amt"])
분석["log_업소수"] = np.log(분석["업소수"])
분석["log_인구"] = np.log(분석["성인인구"])
설명변수 = ["log_업소수", "log_인구"]

모음, 요약 = [], []
for 업종코드, d in 분석.groupby("TP_BUZ_NO"):
    # HC3: 작은 지역일수록 예측이 더 흔들리는 이분산에 대응하는 로버스트 표준오차
    모델 = sm.OLS(d["log_소비"], sm.add_constant(d[설명변수])).fit(cov_type="HC3")
    d = d.assign(모델예측=np.exp(모델.fittedvalues))
    모음.append(d)
    요약.append({"업종": d["TP_BUZ_NM"].iat[0], "지역수": len(d), "R2": 모델.rsquared,
               "점포수계수": 모델.params["log_업소수"], "인구계수": 모델.params["log_인구"]})

결과 = pd.concat(모음, ignore_index=True)
결과["침투지수"] = 결과["amt"] / 결과["모델예측"] * 100
결과["격차금액"] = 결과["모델예측"] - 결과["amt"]
결과["시장백분위"] = 결과.groupby("TP_BUZ_NO")["모델예측"].rank(pct=True) * 100

with pd.option_context("display.float_format", "{:.3f}".format):
    display(pd.DataFrame(요약).sort_values("R2", ascending=False))

## 3. 신뢰 — 모델 오차보다 확실히 낮은가

모델은 원래 어느 정도 틀린다. 업종별로 5-fold 교차검증 예측 오차를 구하고, 부족분이 그 오차의 몇 배인지를 σ로 잰다.

`σ = log(침투지수 / 100) ÷ 업종별 예측 오차`

| σ | 순수 잡음이어도 이 선 아래로 떨어질 확률 |
|---|---|
| −1 | 15.9% |
| −1.5 | 6.7% |
| −2 | 2.3% |

In [ ]:
오차 = {}
for 업종코드, d in 결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    fold오차 = []
    for 학습, 검증 in KFold(5, shuffle=True, random_state=42).split(d):
        m = sm.OLS(d.loc[학습, "log_소비"], sm.add_constant(d.loc[학습, 설명변수])).fit()
        p = m.predict(sm.add_constant(d.loc[검증, 설명변수], has_constant="add"))
        fold오차.append(np.sqrt(np.mean((d.loc[검증, "log_소비"] - p) ** 2)))
    오차[업종코드] = np.mean(fold오차)

결과["시그마"] = np.log(결과["침투지수"] / 100) / 결과["TP_BUZ_NO"].map(오차)

오차표 = (결과.drop_duplicates("TP_BUZ_NO")[["TP_BUZ_NO", "TP_BUZ_NM"]]
        .assign(예측오차=lambda d: d["TP_BUZ_NO"].map(오차))
        .assign(정상범위_하한=lambda d: np.exp(-d["예측오차"]) * 100,
                정상범위_상한=lambda d: np.exp(d["예측오차"]) * 100)
        .sort_values("예측오차"))
with pd.option_context("display.float_format", "{:.2f}".format):
    display(오차표.drop(columns="TP_BUZ_NO"))
print("정상범위 = 모델이 흔히 벗어나는 ±1σ 폭. 이 안의 침투지수는 저침투라 단정하기 어렵다")

## 4. 선정 — 네 기준을 차례로 통과

경계값은 아래 셀 맨 위에 모아 두었다. 바꾸고 다시 실행하면 뒤의 표·그림·저장 파일이 전부 따라 바뀐다.

In [ ]:
# ── 선정 기준 ──
격차경계 = 100        # ① 침투지수 이 값 미만
시장경계 = 50         # ② 업종 내 시장 백분위 이 값 초과 (상위 50%)
신뢰경계 = -1.5       # ③ σ 이 값 이하
규모하한 = 20e8       # ④ 격차금액 이 값 이상 (20억)

m_격차 = 결과["침투지수"] < 격차경계
m_시장 = 결과["시장백분위"] > 시장경계
m_신뢰 = 결과["시그마"] <= 신뢰경계
m_규모 = 결과["격차금액"] >= 규모하한

깔때기 = pd.DataFrame([
    ("전체 조합", len(결과)),
    ("① 격차: 침투지수 < 100", int(m_격차.sum())),
    ("② 시장: 업종 내 상위 50%", int((m_격차 & m_시장).sum())),
    ("③ 신뢰: σ ≤ −1.5", int((m_격차 & m_시장 & m_신뢰).sum())),
    ("④ 규모: 격차금액 ≥ 20억", int((m_격차 & m_시장 & m_신뢰 & m_규모).sum())),
], columns=["단계", "남은 조합"])
display(깔때기)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2), facecolor=바탕)
기본축(ax, 격자축=None)
t = 깔때기.iloc[::-1].reset_index(drop=True)
색 = [주황] + [파랑] * (len(t) - 2) + [흐림]
ax.barh(t["단계"].str.replace("−", "-"), t["남은 조합"], color=색, height=0.62)   # 맑은 고딕엔 긴 마이너스(−) 글꼴이 없다
for i, v in enumerate(t["남은 조합"]):
    ax.text(v, i, f"  {v:,}", va="center", color=잉크, fontsize=10.5, fontweight="bold")
ax.set_xscale("log")
ax.set_xlim(1, t["남은 조합"].max() * 4)
ax.set_xticks([], minor=True)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.set_title("후보 선정 깔때기 (막대 길이는 로그)", color=잉크, fontsize=13, fontweight="bold", loc="left", pad=12)
ax.tick_params(axis="y", labelsize=10.5, colors=보조)
plt.tight_layout()
plt.show()

In [ ]:
후보 = 결과[m_격차 & m_시장 & m_신뢰 & m_규모].copy()

# 같은 지역의 8개 업종 중 몇 개가 100 미만인가 — 지역 전체 약세인지 업종 한정인지 보는 참고값
지역약세 = 결과.groupby("행정구역명")["침투지수"].agg(지역업종수="size", 지역100미만=lambda s: int((s < 100).sum()))
후보 = 후보.merge(지역약세, on="행정구역명")
후보["신뢰등급"] = np.where(후보["시그마"] <= -2, "강", "중")
후보 = 후보.sort_values("격차금액", ascending=False).reset_index(drop=True)

# 거래가 6개월 모두 있었는지 — 극소 거래 조합이 끼지 않았는지 확인
거래월 = (bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & (bc["amt"] > 0)]
        .groupby(["행정구역명", "TP_BUZ_NO"])["STRD_YYMM"].nunique().rename("거래월수"))
후보 = 후보.merge(거래월, on=["행정구역명", "TP_BUZ_NO"], how="left")

print(f"후보 {len(후보)}개 조합 / {후보['행정구역명'].nunique()}개 시군구 / 격차금액 합 {후보['격차금액'].sum() / 1e8:,.0f}억")
print("거래가 6개월 미만인 후보:", int((후보["거래월수"] < 6).sum()), "개")

# σ는 −1.5 경계 근처에 몰려 있어 소수 둘째 자리까지 봐야 한다
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(후보.assign(실제_억=lambda d: d["amt"] / 1e8, 예측_억=lambda d: d["모델예측"] / 1e8,
                      격차_억=lambda d: d["격차금액"] / 1e8)
            [["행정구역명", "TP_BUZ_NM", "업소수", "실제_억", "예측_억", "격차_억",
              "침투지수", "시그마", "신뢰등급", "지역100미만"]])

## 5. 후보 보기

전국 조합 속에서 후보가 어디에 있는지, 그리고 후보끼리 규모와 신뢰가 어떻게 다른지 본다.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7.2), facecolor=바탕)
기본축(ax)

x = 결과["침투지수"].clip(8, 700)
ax.scatter(x, 결과["시장백분위"], s=10, color=흐림, alpha=0.22, edgecolor="none", zorder=2,
           label=f"전체 조합 ({len(결과):,})")
통과 = m_격차 & m_시장
ax.scatter(x[통과], 결과.loc[통과, "시장백분위"], s=13, color=파랑, alpha=0.35, edgecolor="none", zorder=3,
           label=f"① 격차·② 시장 통과 ({int(통과.sum())})")
빠짐 = m_격차 & m_시장 & m_신뢰 & ~m_규모
ax.scatter(결과.loc[빠짐, "침투지수"], 결과.loc[빠짐, "시장백분위"], s=42, marker="x", color=잉크,
           lw=1.3, zorder=4, label=f"③ 신뢰까지 통과, ④ 20억 미만 ({int(빠짐.sum())})")
ax.scatter(후보["침투지수"], 후보["시장백분위"], s=85, color=주황, edgecolor=잉크, lw=1.1, zorder=5,
           label=f"후보 ({len(후보)})")

ax.set_xscale("log")
ax.set_xlim(700, 8)                               # 뒤집어서 오른쪽일수록 격차가 크게
ax.axvline(격차경계, color=잉크, lw=1.1, zorder=1)
ax.axhline(시장경계, color=잉크, lw=1.1, zorder=1)
ax.set_xticks([500, 200, 100, 50, 20, 10])
ax.set_xticklabels(["500", "200", "100", "50", "20", "10"])
ax.set_title("전국 조합 속 후보 위치", color=잉크, fontsize=14, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("침투지수 (로그, 오른쪽일수록 격차 큼) — 100 = 모델 예측대로", color=보조, fontsize=10)
ax.set_ylabel("시장 — 업종 내 백분위 (위일수록 큼)", color=보조, fontsize=10)
ax.legend(frameon=False, loc="lower left", fontsize=9.5, labelcolor=보조)
plt.tight_layout()

# 축 범위가 정해진 뒤에 라벨을 붙여야 겹침 판단이 정확하다 — 격차 큰 후보부터 자리를 잡는다
라벨배치(ax, zip(후보["침투지수"], 후보["시장백분위"]),
       [f"{약칭(g)} {u.replace(' ', '')}" for g, u in zip(후보["행정구역명"], 후보["TP_BUZ_NM"])],
       피할점=zip(결과.loc[빠짐, "침투지수"], 결과.loc[빠짐, "시장백분위"]))
plt.show()

In [ ]:
t = 후보.sort_values("격차금액").reset_index(drop=True)
라벨 = [f"{약칭(g)} {u.replace(' ', '')}" for g, u in zip(t["행정구역명"], t["TP_BUZ_NM"])]
색 = [주황 if s <= -2 else 파랑 for s in t["시그마"]]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.6), facecolor=바탕)
기본축(ax, 격자축="x")
ax.barh(라벨, t["격차금액"] / 1e8, color=색, height=0.62)
for i, (v, s) in enumerate(zip(t["격차금액"] / 1e8, t["시그마"])):
    ax.text(v, i, f"  {v:,.0f}억 · σ {s:.2f}", va="center", color=보조, fontsize=9.5)
ax.set_xlim(0, t["격차금액"].max() / 1e8 * 1.3)
ax.set_title("후보별 격차금액 — 주황은 신뢰 강(σ ≤ -2), 파랑은 중",
             color=잉크, fontsize=13, fontweight="bold", loc="left", pad=12)
ax.set_xlabel("격차금액 (억원, 6개월)", color=보조, fontsize=10)
ax.tick_params(axis="y", labelsize=10, colors=보조)
plt.tight_layout()
plt.show()

In [ ]:
# 이식 후 확인용
print(f"BC {len(bc):,}행 · 업종 {len(분석업종)}개 · 지역 {업소['행정구역명'].nunique()}개")


업종별 회귀 엔진

In [ ]:
행 = []
for 코드, d in 분석.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)          # ← 정렬을 새로 하지 마세요 (순서가 폴드를 결정합니다)
    X = sm.add_constant(np.column_stack([np.log(d["업소수"]), np.log(d["성인인구"])]))
    y = np.log(d["amt"].values)
    f = sm.OLS(y, X).fit(cov_type="HC3")

    # 5-fold CV 예측오차 → σ의 분모 (난수 없이 결정적으로)
    n = len(d)
    oof = np.full(n, np.nan)
    for i in range(5):
        te = np.arange(n)[i::5]
        tr = np.setdiff1d(np.arange(n), te)
        oof[te] = X[te] @ np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]
    cv오차 = np.std(y - oof, ddof=1)

    t = d.copy()
    t["모델예측"] = np.exp(f.fittedvalues)
    t["침투지수"] = t["amt"] / t["모델예측"] * 100
    t["격차금액"] = t["모델예측"] - t["amt"]
    t["시장백분위"] = t["모델예측"].rank(pct=True) * 100
    t["시그마"] = np.log(t["침투지수"] / 100) / cv오차
    t["CV오차"] = cv오차; t["R2"] = f.rsquared
    행.append(t)

결과 = pd.concat(행, ignore_index=True)
결과["log_업소수"], 결과["log_인구"] = np.log(결과["업소수"]), np.log(결과["성인인구"])
결과["log_소비"] = np.log(결과["amt"])
print(f"{len(결과):,}개 조합 · 침투지수 {결과['침투지수'].min():.1f}~{결과['침투지수'].max():.1f}")


그림1: 로그 변환 근거 (12쪽)

In [ ]:
def 잔차(d, 로그):
    X = np.column_stack([np.ones(len(d)),
         np.log(d["업소수"]) if 로그 else d["업소수"].astype(float),
         np.log(d["성인인구"]) if 로그 else d["성인인구"]])
    y = np.log(d["amt"].values) if 로그 else d["amt"].values
    return y - X @ np.linalg.lstsq(X, y, rcond=None)[0]

쏠림 = []
for _, d in 결과.groupby("TP_BUZ_NO"):
    큰 = (d["성인인구"] >= d["성인인구"].quantile(0.9)).values
    쏠림.append({"업종": d["TP_BUZ_NM"].iat[0].replace(" ", ""),
        **{n: np.abs(잔차(d, lg))[큰].mean() / np.abs(잔차(d, lg))[~큰].mean()
           for n, lg in [("원자료", False), ("로그", True)]}})
쏠림 = pd.DataFrame(쏠림).sort_values("원자료", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
s = 결과["amt"] / 1e8
ax[0].hist(np.log(s), bins=55, color=C1, edgecolor="white", lw=.4)
ax[0].set(xlabel="log(BC 소비)", ylabel="지역·업종 조합 수",
          title=f"로그 변환 후 분포는 대칭에 가깝다  (왜도 {stats.skew(s):.2f} → {stats.skew(np.log(s)):.2f})")

p = np.arange(len(쏠림))
ax[1].barh(p + .19, 쏠림["원자료"], height=.36, color=FAINT, label="원자료")
ax[1].barh(p - .19, 쏠림["로그"],  height=.36, color=C1, label="로그 변환")
ax[1].axvline(1, color=MUTED, ls="--", lw=1)
ax[1].set(yticks=p, yticklabels=쏠림["업종"],
          xlabel="상위 10% 대도시 평균 오차 ÷ 나머지 지역 평균 오차",
          title="원자료로 적합하면 모델이 대도시에 맞춰진다")
ax[1].legend(frameon=False, loc="lower right")
fig.tight_layout(); 저장(fig, "12_로그변환"); plt.show()


모델 설명력 표

In [ ]:
설명력 = (결과.groupby("TP_BUZ_NM")
    .agg(지역수=("amt", "size"), R2=("R2", "first"), CV오차=("CV오차", "first"))
    .sort_values("R2", ascending=False))
설명력["정상범위_하한"] = (np.exp(-설명력["CV오차"]) * 100).round(0)
설명력["정상범위_상한"] = (np.exp(설명력["CV오차"]) * 100).round(0)
print(설명력.round(3).to_string())
print(f"\n가중평균 R² {np.average(설명력['R2'], weights=설명력['지역수']):.3f}")


그림2: 예측 vs 실제

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6))
ax.scatter(결과["모델예측"]/1e8, 결과["amt"]/1e8, s=13, alpha=.35,
           color=C1, edgecolor="none")
lim = [결과["모델예측"].min()/1e8*0.7, 결과["모델예측"].max()/1e8*1.3]
ax.plot(lim, lim, color=INK, lw=1.4)
후보키 = None  # 셀 9 이후 다시 실행하면 후보가 강조됩니다
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim,
       xlabel="모델 예측 BC 소비 (억원)", ylabel="실제 BC 소비 (억원)",
       title="점포 수와 인구로 예측한 값 vs 실제  (대각선 = 침투지수 100)")
ax.text(.03, .95, f"상관 {np.corrcoef(결과['모델예측'], 결과['amt'])[0,1]:.3f}",
        transform=ax.transAxes, color=MUTED, va="top")
fig.tight_layout(); 저장(fig, "13_예측vs실제"); plt.show()


검증

In [ ]:
def 적합(X, y): return np.linalg.lstsq(X, y, rcond=None)[0]
def 설계(d): return np.column_stack([np.ones(len(d)), np.log(d["업소수"]), np.log(d["성인인구"])])

rng = np.random.default_rng(42); 검증행 = []
결과["시도"] = 결과["행정구역명"].str.split().str[0]
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True); X, y = 설계(d), np.log(d["amt"].values)
    idx = rng.permutation(len(d)); oof = np.full(len(d), np.nan)
    for f_ in np.array_split(idx, 5):
        oof[f_] = X[f_] @ 적합(X[np.setdiff1d(idx, f_)], y[np.setdiff1d(idx, f_)])
    grp = np.full(len(d), np.nan)
    for s in d["시도"].unique():
        te = (d["시도"] == s).values
        if te.sum() < len(d): grp[te] = X[te] @ 적합(X[~te], y[~te])
    ok = ~np.isnan(grp)
    r2 = lambda yy, pp: 1 - ((yy-pp)**2).sum()/((yy-yy.mean())**2).sum()
    corr = np.corrcoef(np.log(d["업소수"]), np.log(d["성인인구"]))[0,1]
    검증행.append(dict(업종=d["TP_BUZ_NM"].iat[0].replace(" ",""),
        R2학습=d["R2"].iat[0], R2_OOF=r2(y, oof), R2_시도홀드=r2(y[ok], grp[ok]),
        VIF=1/(1-corr**2)))
검증 = pd.DataFrame(검증행)
검증["과적합"] = 검증["R2학습"] - 검증["R2_OOF"]
print(검증.round(4).to_string(index=False))

# 컷오프 민감도
def 뽑기(s_cut, 격차억, 시장=50):
    m = ((결과["침투지수"]<100) & (결과["시장백분위"]>=시장)
         & (결과["시그마"]<=s_cut) & (결과["격차금액"]>=격차억*1e8))
    return set(map(tuple, 결과.loc[m, ["행정구역명","TP_BUZ_NO"]].values))
격자 = pd.DataFrame({f"{g}억": {s: len(뽑기(s, g)) for s in [-1.3,-1.4,-1.5,-1.6,-1.7,-2.0]}
                    for g in [10,20,30,50]})
print("\n컷오프 민감도 (후보 수)"); print(격자.to_string())


그림3: 검증 종합

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
p = np.arange(len(검증))
ax[0].barh(p+.19, 검증["R2학습"], height=.36, color=FAINT, label="학습")
ax[0].barh(p-.19, 검증["R2_시도홀드"], height=.36, color=C1, label="시도 홀드아웃")
ax[0].set(yticks=p, yticklabels=검증["업종"], xlim=(0,1), xlabel="R²",
          title="시도를 통째로 빼고 예측해도 설명력이 유지된다")
ax[0].legend(frameon=False, loc="lower right")

im = ax[1].imshow(격자.values, cmap="Purples", aspect="auto")
ax[1].set(xticks=range(len(격자.columns)), xticklabels=격자.columns,
          yticks=range(len(격자.index)), yticklabels=[f"σ ≤ {v}" for v in 격자.index],
          title="컷오프를 흔들었을 때 후보 수")
for i in range(격자.shape[0]):
    for j in range(격자.shape[1]):
        ax[1].text(j, i, 격자.values[i,j], ha="center", va="center",
                   color="white" if 격자.values[i,j] > 18 else INK, fontsize=10)
ax[1].add_patch(plt.Rectangle((0.5, 1.5), 1, 1, fill=False, edgecolor=C2, lw=2.5))
fig.tight_layout(); 저장(fig, "14_검증"); plt.show()


후보 12개

In [ ]:
단계 = [("전체 시장", 결과),
        ("예상보다 적게 결제", 결과[결과["침투지수"]<100]),
        ("업종 내 큰 시장", 결과[(결과["침투지수"]<100)&(결과["시장백분위"]>=50)]),
        ("우연이 아님 (σ≤−1.5)", 결과[(결과["침투지수"]<100)&(결과["시장백분위"]>=50)&(결과["시그마"]<=-1.5)])]
후보 = 단계[-1][1][단계[-1][1]["격차금액"]>=20e8].sort_values("격차금액", ascending=False)
단계.append(("격차 20억 이상", 후보))
for n, d in 단계: print(f"  {n:22s} {len(d):5,}개")
print(f"\n{len(후보)}개 · {후보['행정구역명'].nunique()}개 시군구 · {후보['격차금액'].sum()/1e8:,.0f}억")
후보키 = set(map(tuple, 후보[["행정구역명","TP_BUZ_NO"]].values))


그림4: 깔때기

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.4))
n = [len(d) for _, d in 단계]; 이름 = [x[0] for x in 단계]
for i, (nm, v) in enumerate(zip(이름, n)):
    w = np.log10(v+1)/np.log10(n[0]+1)
    ax.barh(i, w, height=.62, color=C1 if i < len(n)-1 else C2,
            alpha=.35 + .13*i, edgecolor="white", lw=2)
    ax.text(w+.015, i, f"{v:,}개", va="center", color=INK, fontweight="bold")
ax.set(yticks=range(len(n)), yticklabels=이름, xlim=(0,1.18), xticks=[])
ax.invert_yaxis(); ax.grid(False)
for s in ["top","right","bottom","left"]: ax.spines[s].set_visible(False)
ax.set_title(f"1,813개에서 {len(후보)}개로 — 네 조건을 모두 통과해야 남는다", pad=14)
fig.tight_layout(); 저장(fig, "15_깔때기"); plt.show()


그림5: 전국 지도

In [ ]:
import json, urllib.request
경로 = "data/cache/korea_sigungu.json"; os.makedirs("data/cache", exist_ok=True)
if not os.path.exists(경로):
    url = ("https://raw.githubusercontent.com/southkorea/southkorea-maps/master/"
           "kostat/2018/json/skorea-municipalities-2018-geo.json")
    with urllib.request.urlopen(url, timeout=30) as r: open(경로,"wb").write(r.read())
지도 = json.load(open(경로, encoding="utf-8"))

후보지역 = 후보.groupby("행정구역명")["격차금액"].sum() / 1e8
fig, ax = plt.subplots(figsize=(7.4, 9))
for feat in 지도["features"]:
    g = feat["geometry"]
    폴리 = g["coordinates"] if g["type"]=="Polygon" else [c[0] for c in g["coordinates"]]
    for poly in ([폴리[0]] if g["type"]=="Polygon" else 폴리):
        xy = np.array(poly)
        ax.fill(xy[:,0], xy[:,1], facecolor="#F4F4F8", edgecolor="white", lw=.4)
좌표 = {}   # 후보 시군구 중심점
for feat in 지도["features"]:
    nm = feat["properties"].get("name", "")
    for z in 후보지역.index:
        if nm and nm in z:
            g = feat["geometry"]
            poly = g["coordinates"][0] if g["type"]=="Polygon" else g["coordinates"][0][0]
            좌표[z] = np.array(poly).mean(axis=0)
for z, (x, y) in 좌표.items():
    v = 후보지역[z]
    ax.scatter(x, y, s=v*3.2, color=C1, alpha=.55, edgecolor=C1, lw=1.4, zorder=3)
    ax.annotate(f"{z.split()[-1]}\n{v:,.0f}억", (x, y), fontsize=9, ha="center",
                va="center", color=INK, fontweight="bold", zorder=4)
ax.set_aspect("equal"); ax.axis("off"); ax.grid(False)
ax.set_title(f"후보 {len(후보)}개 · {len(후보지역)}개 시군구 · 격차 {후보['격차금액'].sum()/1e8:,.0f}억", pad=10)
fig.tight_layout(); 저장(fig, "15_지도"); plt.show()



유형화 7/4/1

In [ ]:
장보기, 외식 = ["4020","4004","4010"], ["8001","8002","8003","8004","8005","8006","8021","8301"]
b = bc[bc["GENDER_CD"].isin(["1","2","3"]) & bc["행정구역명"].isin(결과["행정구역명"].unique())]
성인 = 결과.groupby("행정구역명")["성인인구"].first()
def 한명당지수(코드들=None):
    sub = b if 코드들 is None else b[b["TP_BUZ_NO"].isin(코드들)]
    s = sub.groupby("행정구역명")["amt"].sum().reindex(성인.index).fillna(0)
    return (s/성인) / (s.sum()/성인.sum()) * 100

지역지수, 장보기지수, 외식지수, 대형지수 = 한명당지수(), 한명당지수(장보기), 한명당지수(외식), 한명당지수(["4004"])

진단 = 후보[["행정구역명","TP_BUZ_NO","TP_BUZ_NM","침투지수","격차금액","시그마"]].copy()
진단["지역지수"] = 지역지수.reindex(진단["행정구역명"]).values
진단["카테고리지수"] = [장보기지수[z] if c in 장보기 else 외식지수[z]
                   for z, c in zip(진단["행정구역명"], 진단["TP_BUZ_NO"])]
진단["대형할인점"] = 대형지수.reindex(진단["행정구역명"]).values
진단["카테고리÷지역"] = 진단["카테고리지수"] / 진단["지역지수"] * 100

def 유형판정(r):
    if r["지역지수"] >= 100: return 3                    # 지역은 강한데 업종만 약함
    if r["지역지수"] < 80 and r["카테고리지수"] < 80: return 1   # 3층 모두 낮음
    return 2                                            # 지역·카테고리는 정상 수준
진단["유형"] = 진단.apply(유형판정, axis=1)
진단["유형명"] = 진단["유형"].map(유형명)

print(진단.sort_values(["유형","격차금액"], ascending=[True,False]).round(1).to_string(index=False))
print()
for t in [1,2,3]:
    d = 진단[진단["유형"]==t]
    print(f"{유형명[t]}: {len(d)}개 · {d['격차금액'].sum()/1e8:,.0f}억 · "
          f"지역지수 {d['지역지수'].min():.0f}~{d['지역지수'].max():.0f} · "
          f"대형할인점 {d['대형할인점'].min():.0f}~{d['대형할인점'].max():.0f}")


셀 13 — 🖼 그림6: 유형 산점도 ★핵심 (16쪽)



In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 6.4))
ax.axvline(100, color=MUTED, ls="--", lw=1)
ax.axhline(100, color=MUTED, ls="--", lw=1)
for t in [1,2,3]:
    d = 진단[진단["유형"]==t]
    ax.scatter(d["지역지수"], d["침투지수"], s=d["격차금액"]/1e8*2.6,
               color=유형색[t], alpha=.65, edgecolor="white", lw=1.6,
               label=f"{유형명[t]}  {len(d)}개 · {d['격차금액'].sum()/1e8:,.0f}억", zorder=3)
for r in 진단.itertuples():
    ax.annotate(f"{r.행정구역명.split()[-1]} {r.TP_BUZ_NM.replace(' ','')}",
                (r.지역지수, r.침투지수), fontsize=8.5, color=INK,
                xytext=(0, -16), textcoords="offset points", ha="center")
ax.set(xlabel="지역 전체 BC 이용 수준 (1인당, 전국=100)",
       ylabel="해당 업종 침투지수 (예측 대비, 100=정상)",
       title="가로축이 지역의 상태, 세로축이 업종의 상태 — 두 축이 유형을 가른다")
ax.legend(frameon=False, loc="upper left", fontsize=10)
ax.text(101, 12, "→ 지역은 정상", color=MUTED, fontsize=9)
fig.tight_layout(); 저장(fig, "16_유형화"); plt.show()


셀 14 — 🖼 그림7: 유형별 지표 비교 (17쪽)



In [ ]:
지표 = ["침투지수","지역지수","카테고리지수","대형할인점"]
fig, ax = plt.subplots(1, 4, figsize=(15, 4.2), sharey=False)
for k, col in enumerate(지표):
    for t in [1,2,3]:
        v = 진단.loc[진단["유형"]==t, col]
        ax[k].scatter([t]*len(v), v, s=52, color=유형색[t], alpha=.75,
                      edgecolor="white", lw=1.2, zorder=3)
        ax[k].plot([t-.26,t+.26], [v.mean()]*2, color=유형색[t], lw=2.4)
    ax[k].axhline(100, color=MUTED, ls="--", lw=1)
    ax[k].set(xticks=[1,2,3], xticklabels=["①","②","③"], xlim=(.5,3.5), title=col)
ax[0].set_ylabel("지수 (100 기준)")
fig.suptitle("유형을 가르는 건 대형할인점 지수 — ①은 전부 100 미만, ②③은 93 이상", y=1.02)
fig.tight_layout(); 저장(fig, "17_유형지표"); plt.show()


셀 15 — 🖼 그림8: 건수 vs 건당 (18쪽)



In [ ]:
건수합 = b.groupby(["행정구역명","TP_BUZ_NO"])["cnt"].sum()
건수지수 = {}
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    X = 설계(d); y = np.log([건수합[(z,코드)] for z in d["행정구역명"]])
    건수지수.update({(z,코드): v for z, v in zip(d["행정구역명"], np.exp(y - X@적합(X,y))*100)})
진단["건수지수"] = [건수지수[(z,c)] for z,c in zip(진단["행정구역명"], 진단["TP_BUZ_NO"])]
진단["건당지수"] = 진단["침투지수"] / 진단["건수지수"] * 100

fig, ax = plt.subplots(figsize=(8.4, 5.6))
ax.axhline(100, color=MUTED, ls="--", lw=1); ax.axvline(100, color=MUTED, ls="--", lw=1)
for t in [1,2,3]:
    d = 진단[진단["유형"]==t]
    ax.scatter(d["건수지수"], d["건당지수"], s=d["격차금액"]/1e8*2.6, color=유형색[t],
               alpha=.68, edgecolor="white", lw=1.6, label=유형명[t], zorder=3)
for r in 진단.itertuples():
    ax.annotate(r.행정구역명.split()[-1], (r.건수지수, r.건당지수), fontsize=8.5,
                xytext=(0,-15), textcoords="offset points", ha="center", color=INK)
ax.set(xlim=(0,110), xlabel="건수지수 — 얼마나 자주 결제하나", ylabel="건당지수 — 한 번에 얼마나 쓰나",
       title="부족분은 전부 결제 횟수에서 나온다 (건당은 정상)")
ax.legend(frameon=False, loc="lower right", fontsize=10)
fig.tight_layout(); 저장(fig, "18_건수건당"); plt.show()


셀 16 — 🖼 그림9: 연령 히트맵 (19쪽)



In [ ]:
성인CD = ["2","3","4","5","6"]; 라벨 = {"2":"20대","3":"30대","4":"40대","5":"50대","6":"60대+"}
연령인구 = (pop[pop["AGE_CD"].isin(성인CD)]
    .groupby(["행정구역명","AGE_CD","STRD_YYMM"])["인구"].sum()
    .groupby(["행정구역명","AGE_CD"]).mean())
전국인구 = 연령인구.groupby("AGE_CD").sum()
bi = bc[bc["GENDER_CD"].isin(["1","2"]) & bc["AGE_CD"].isin(성인CD)
        & bc["TP_BUZ_NO"].isin(분석업종) & bc["행정구역명"].isin(성인.index)]
지, 전 = bi.groupby(["행정구역명","TP_BUZ_NO","AGE_CD"])["amt"].sum(), bi.groupby(["TP_BUZ_NO","AGE_CD"])["amt"].sum()

연령 = {}
for z, c in zip(진단["행정구역명"], 진단["TP_BUZ_NO"]):
    지1 = pd.Series({a: 지.get((z,c,a),0)/연령인구[(z,a)] for a in 성인CD})
    지평 = sum(지.get((z,c,a),0) for a in 성인CD)/sum(연령인구[(z,a)] for a in 성인CD)
    전1 = pd.Series({a: 전[(c,a)]/전국인구[a] for a in 성인CD})
    연령[f"{z.split()[-1]} {진단.set_index(['행정구역명','TP_BUZ_NO']).loc[(z,c),'TP_BUZ_NM'].replace(' ','')}"] = \
        (지1/지평)/(전1/(전[c].sum()/전국인구.sum()))*100
A = pd.DataFrame(연령).T.rename(columns=라벨)

fig, ax = plt.subplots(figsize=(7.4, 6.4))
im = ax.imshow(A.values, cmap="PuOr_r", vmin=60, vmax=140, aspect="auto")
ax.set(xticks=range(5), xticklabels=A.columns, yticks=range(len(A)), yticklabels=A.index)
for i in range(len(A)):
    for j in range(5):
        ax.text(j, i, f"{A.values[i,j]:.0f}", ha="center", va="center",
                fontsize=9.5, color=INK, fontweight="bold")
ax.grid(False); plt.colorbar(im, ax=ax, label="연령지수 (전국 동업종=100)")
ax.set_title("세대 타깃이 가능한 곳은 진주·서초뿐 (나머지는 전 연령)", pad=12)
fig.tight_layout(); 저장(fig, "19_연령"); plt.show()
print(A.round(0).to_string())


셀 17 — 🖼 그림10: 서초 심층 (20쪽)



In [ ]:
서울 = 결과[(결과["행정구역명"].str.startswith("서울")) & (결과["TP_BUZ_NO"]=="4020")].copy()
서울["대형할인점"] = 대형지수.reindex(서울["행정구역명"]).values
서울 = 서울.sort_values("침투지수")
색 = [C3 if "서초" in z else FAINT for z in 서울["행정구역명"]]

fig, ax = plt.subplots(1, 2, figsize=(14, 5.4))
p = np.arange(len(서울))
ax[0].barh(p, 서울["침투지수"], color=색, height=.72)
ax[0].axvline(서울["침투지수"].median(), color=MUTED, ls="--", lw=1)
ax[0].set(yticks=p, yticklabels=[z.split()[-1] for z in 서울["행정구역명"]],
          xlabel="슈퍼마켓 침투지수",
          title=f"서초는 서울 25개구 중 최저 (28.2 · 중앙값 {서울['침투지수'].median():.0f})")
ax[0].text(서울["침투지수"].median()+2, 1, "중앙값", color=MUTED, fontsize=9)

ax[1].scatter(서울["대형할인점"], 서울["침투지수"], s=70, color=FAINT,
              edgecolor="white", lw=1.4, zorder=3)
s초 = 서울[서울["행정구역명"].str.contains("서초")]
ax[1].scatter(s초["대형할인점"], s초["침투지수"], s=190, color=C3,
              edgecolor="white", lw=2, zorder=4)
ax[1].annotate("서초", (s초["대형할인점"].iat[0], s초["침투지수"].iat[0]),
               xytext=(8,8), textcoords="offset points", fontweight="bold", color=C3)
ax[1].set(xlabel="대형할인점 이용지수 (1인당, 전국=100)", ylabel="슈퍼마켓 침투지수",
          title="장보기 돈이 대형마트로 가 있다")
fig.tight_layout(); 저장(fig, "20_서초"); plt.show()


셀 18 — 대조군과 최소 탐지 기준 (23쪽)



In [ ]:
시범 = [("대전광역시 서구","8001"),("대전광역시 서구","8006"),
        ("경상남도 진주시","4020"),("서울특별시 서초구","4020")]
R = 결과.set_index(["행정구역명","TP_BUZ_NO"])
for z, c in 시범:
    p = R.loc[(z,c)]
    d = 결과[(결과["TP_BUZ_NO"]==c) & ~결과.set_index(["행정구역명","TP_BUZ_NO"]).index.isin(후보키)].copy()
    d["규모비"] = d["모델예측"]/p["모델예측"]
    cand = d[(d["규모비"].between(0.7,1.3)) & (d["시그마"].between(-0.5,0.5))].copy()
    cand["점수"] = np.log(cand["규모비"]).abs() + cand["시그마"].abs()*0.5
    print(f"[{z} {p['TP_BUZ_NM']}] 대조군: "
          + ", ".join(cand.nsmallest(4,"점수")["행정구역명"].str.split().str[-1]))

# 월별 지수의 흔들림 = 최소 탐지 기준
월 = sorted(b["STRD_YYMM"].astype(str).unique())
단면 = 결과.set_index(["행정구역명","TP_BUZ_NO"])[["업소수","성인인구"]]
m = b.groupby(["행정구역명","TP_BUZ_NO","STRD_YYMM"])[["amt","cnt"]].sum().reset_index()
m = m.join(단면, on=["행정구역명","TP_BUZ_NO"]).dropna(); m = m[(m["amt"]>0)&(m["cnt"]>0)]
시리즈 = {}
for (c, ym), d in m.groupby(["TP_BUZ_NO","STRD_YYMM"]):
    X = 설계(d)
    for col, nm in [("amt","침투지수"),("cnt","건수지수")]:
        y = np.log(d[col].values)
        for z, v in zip(d["행정구역명"], np.exp(y - X@적합(X,y))*100):
            시리즈.setdefault((z,c,nm), {})[str(ym)] = v
for z, c in 시범:
    for nm in ["침투지수","건수지수"]:
        s = pd.Series(시리즈[(z,c,nm)]).reindex(월)
        print(f"  {z.split()[-1]:6s} {nm} 월별 SD {s.std():.2f} → 분기 최소 탐지 {1.96*s.std()/np.sqrt(3):.1f}p")


In [ ]:
s초 = 서울[서울["행정구역명"].str.contains("서초")]
서울["구"] = 서울["행정구역명"].str.split().str[-1]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))     # 슬라이드 비율 3.95에 맞춤

a = ax[0]; med = np.median(서울["침투지수"].values)
기타 = 서울[서울["구"] != "서초구"]["침투지수"].values
a.scatter(기타, np.zeros_like(기타) + np.random.default_rng(3).normal(0, .05, len(기타)),
          s=95, color=FAINT, edgecolor="white", lw=1.4, zorder=3)
a.scatter(s초["침투지수"], [0], s=210, color=C3, edgecolor="white", lw=2, zorder=5)
a.annotate("서초  28.2", (s초["침투지수"].iat[0], 0), xytext=(0, 26),
           textcoords="offset points", ha="center", fontsize=13, fontweight="bold", color=C3)
a.axvline(med, color=MUTED, ls="--", lw=1.2)
a.annotate(f"중앙값 {med:.0f}", (med, 0), xytext=(6, -32), textcoords="offset points",
           fontsize=10.5, color=MUTED)
for 구 in ["금천구", "종로구"]:
    r = 서울[서울["구"] == 구]
    a.annotate(구, (r["침투지수"].iat[0], 0), xytext=(0, -34), textcoords="offset points",
               ha="center", fontsize=9.5, color=MUTED)
a.set_ylim(-.34, .34); a.set_yticks([]); a.set_xlim(10, 135)
a.set_xlabel("슈퍼마켓 침투지수"); a.grid(axis="y", visible=False)
a.set_title("서울 25개 자치구 중 서초가 가장 낮다", fontsize=12.5, color=INK, pad=12)

c = ax[1]
c.scatter(서울["대형할인점"], 서울["침투지수"], s=80, color=FAINT, edgecolor="white", lw=1.3, zorder=3)
c.scatter(s초["대형할인점"], s초["침투지수"], s=200, color=C3, edgecolor="white", lw=2, zorder=4)
c.annotate("서초", (s초["대형할인점"].iat[0], s초["침투지수"].iat[0]), xytext=(10, 8),
           textcoords="offset points", fontsize=12, fontweight="bold", color=C3)
for 구, dx, dy in [("강남구", 10, 4), ("송파구", 10, -12)]:
    r = 서울[서울["구"] == 구]
    c.annotate(구, (r["대형할인점"].iat[0], r["침투지수"].iat[0]), xytext=(dx, dy),
               textcoords="offset points", fontsize=9.5, color=MUTED)
c.set(xlabel="대형할인점 이용지수 (1인당, 전국=100)", ylabel="슈퍼마켓 침투지수")
c.set_title("대형마트는 많이 쓰는데 슈퍼마켓만 낮다", fontsize=12.5, color=INK, pad=12)

fig.tight_layout(); 저장(fig, "20_서초_슬라이드"); plt.show()
